<a href="https://colab.research.google.com/github/pratham-239/ML-Project/blob/main/MLL_Project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
import glob, os

from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [ ]:
FOLDER = "/content/drive/MyDrive/amazon_csvs"      # change if your folder name is different

files = sorted(glob.glob(FOLDER + "/*.csv"))
print("Files found:", len(files))
print(files[:3])

Files found: 140
['/content/drive/MyDrive/amazon_csvs/Air Conditioners.csv', '/content/drive/MyDrive/amazon_csvs/All Appliances.csv', '/content/drive/MyDrive/amazon_csvs/All Books.csv']


In [ ]:
def clean_number(x):
    x = str(x).replace("₹", "").replace(",", "").strip()
    return pd.to_numeric(x, errors="coerce")


def clean_one_file(path):
    part = pd.read_csv(path)

    # Make sure the file has the columns we need
    needed = ["name", "ratings", "no_of_ratings", "discount_price", "actual_price"]
    for col in needed:
        if col not in part.columns:
            raise ValueError("missing column: " + col)

    # Turn text like "₹10,999" into real numbers
    for col in ["discount_price", "actual_price", "ratings", "no_of_ratings"]:
        part[col] = part[col].apply(clean_number)

    # Remove products with no price, or a price of 0
    part = part.dropna(subset=["discount_price"])
    part = part[part["discount_price"] > 0]

    # Category = the file's name, e.g. "laptops"
    part["category"] = os.path.basename(path).replace(".csv", "")

    # Keep only the useful columns
    return part[["name", "category", "ratings", "no_of_ratings", "discount_price", "actual_price"]]

In [ ]:
test = clean_one_file(files[0])
print(test.shape)
test.head()

(457, 6)


,name,category,ratings,no_of_ratings,discount_price,actual_price
0,Lloyd 1.5 Ton 3 Star Inverter Split Ac (5 In 1...,Air Conditioners,4.2,2255.0,32999.0,58990.0
1,LG 1.5 Ton 5 Star AI DUAL Inverter Split AC (C...,Air Conditioners,4.2,2948.0,46490.0,75990.0
2,LG 1 Ton 4 Star Ai Dual Inverter Split Ac (Cop...,Air Conditioners,4.2,1206.0,34490.0,61990.0
3,LG 1.5 Ton 3 Star AI DUAL Inverter Split AC (C...,Air Conditioners,4.0,69.0,37990.0,68990.0
4,Carrier 1.5 Ton 3 Star Inverter Split AC (Copp...,Air Conditioners,4.1,630.0,34490.0,67790.0


In [ ]:
tables = []        # a list of cleaned tables
skipped = []       # files that had a problem

for path in files:
    try:
        tables.append(clean_one_file(path))
    except Exception as e:
        skipped.append((os.path.basename(path), str(e)))

print("Cleaned:", len(tables), "files")
print("Skipped:", len(skipped), "files")
for name, reason in skipped:
    print("  ", name, "->", reason)

Cleaned: 140 files
Skipped: 0 files


In [ ]:
all_df = pd.concat(tables, ignore_index=True)

print("Total products:", all_df.shape)
print("Number of categories:", all_df["category"].nunique())
all_df["category"].value_counts().head(10)

Total products: (980844, 6)
Number of categories: 113


/tmp/ipykernel_3576/1858038313.py:1: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  all_df = pd.concat(tables, ignore_index=True)


,count
category,
Amazon-Products,490422
Fashion and Silver Jewellery,18678
Jewellery,18508
Clothing,18303
Ethnic Wear,18287
Bags and Luggage,18283
Western Wear,18021
Handbags and Clutches,17741
T-shirts and Polos,17610


In [ ]:
print("Before:", len(all_df))
all_df = all_df.drop_duplicates(subset=["name", "discount_price"])
print("After:", len(all_df))

Before: 980844
After: 388686


In [ ]:
all_df["ratings"] = all_df["ratings"].fillna(all_df["ratings"].median())
all_df["no_of_ratings"] = all_df["no_of_ratings"].fillna(0)

In [ ]:
print(all_df.shape)
print(all_df.isnull().sum())
all_df["discount_price"].describe()

(388686, 6)
name              0
category          0
ratings           0
no_of_ratings     0
discount_price    0
actual_price      0
dtype: int64


,discount_price
count,3.886860e+05
mean,2.685252e+03
std,9.868176e+03
min,8.000000e+00
25%,3.800000e+02
50%,6.680000e+02
75%,1.399000e+03
max,1.249990e+06


In [ ]:
all_df.to_csv("/content/drive/MyDrive/amazon_csvs/all_products_clean.csv", index=False)
print("Saved!")

Saved!


START

In [ ]:
all_df = all_df.dropna(subset=["name"])

if len(all_df) > 100000:
    data = all_df.sample(100000, random_state=42)
else:
    data = all_df.copy()

data = data.reset_index(drop=True)
print(data.shape)

(100000, 6)


In [ ]:
def add_features(data):
    lower = data["name"].str.lower()

    data["is_renewed"] = lower.str.contains("renewed").astype(int)
    data["name_length"] = data["name"].str.len()
    data["has_gb"] = lower.str.contains("gb").astype(int)

    # RAM and storage (cut off marketing text after "upto")
    short = lower.str.split("upto|up to", regex=True).str[0]
    ram_a = short.str.extract(r"(\d+)\s*gb\s*ram")[0]
    sto_a = short.str.extract(r"(\d+)\s*gb\s*(?:storage|rom|ssd)")[0]
    pair = short.str.extract(r"(\d+)\s*gb\s*,\s*(\d+)\s*gb")

    data["ram_gb"] = pd.to_numeric(ram_a.fillna(pair[0]), errors="coerce").fillna(0)
    data["storage_gb"] = pd.to_numeric(sto_a.fillna(pair[1]), errors="coerce").fillna(0)

    # Brand = first word of the name
    data["brand"] = data["name"].str.split().str[0].str.title()
    return data

data = add_features(data)
data[["name", "brand", "ram_gb", "storage_gb"]].head()

,name,brand,ram_gb,storage_gb
0,WAIMEA Men Coal Black Sturdy Slim Fit Stretcha...,Waimea,0.0,0.0
1,MAKE IN INDIA //All Purpose 3 in 1 Fan Copper ...,Make,0.0,0.0
2,Lymio Casual Shirt for Men|| Shirt for Men|| M...,Lymio,0.0,0.0
3,FASHOLIC USA Denim Unisex Baseball Caps Casual...,Fasholic,0.0,0.0
4,Baidyanath Sanjivani Bati (40Tablet) PACK OF 3,Baidyanath,0.0,0.0


In [ ]:
# Brands: keep the 50 most common, the rest become "Other"
top_brands = list(data["brand"].value_counts().head(50).index)
data["brand_group"] = data["brand"].where(data["brand"].isin(top_brands), "Other")

brand_df = pd.get_dummies(data["brand_group"], prefix="brand", dtype=int)

# Categories: one yes/no column per category (the file names)
category_list = sorted(data["category"].unique())
cat_df = pd.get_dummies(data["category"], prefix="cat", dtype=int)

print(brand_df.shape, cat_df.shape)

(100000, 51) (100000, 11)


In [ ]:
from sklearn.feature_extraction.text import CountVectorizer

word_tool = CountVectorizer(binary=True, max_features=500,
                            stop_words="english", dtype=np.int8)
word_table = word_tool.fit_transform(data["name"])

word_names = ["word_" + w for w in word_tool.get_feature_names_out()]
words_df = pd.DataFrame(word_table.toarray(), columns=word_names, index=data.index)

print(words_df.shape)

(100000, 500)


In [ ]:
basic = ["ratings", "no_of_ratings", "is_renewed",
         "name_length", "has_gb", "ram_gb", "storage_gb"]

X_big = pd.concat([data[basic], brand_df, cat_df, words_df], axis=1)
y_big = np.log1p(data["discount_price"])

print("Clues table:", X_big.shape)
print("Missing values:", X_big.isnull().sum().sum())

Clues table: (100000, 569)
Missing values: 0


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor

Xb_train, Xb_test, yb_train, yb_test = train_test_split(
    X_big, y_big, test_size=0.2, random_state=42
)

big_model = RandomForestRegressor(
    n_estimators=60, min_samples_leaf=2, n_jobs=-1, random_state=42
)
big_model.fit(Xb_train, yb_train)
print("Training finished")

Training finished


In [ ]:
from sklearn.metrics import r2_score

real = np.expm1(yb_test)
guesses_log = big_model.predict(Xb_test)
guesses = np.expm1(guesses_log)

def percent_mistake(real, guess):
    return ((real - guess).abs() / real * 100).median()

lazy = np.expm1(yb_train.median())

print("Lazy baseline  -> typical error:", round(percent_mistake(real, lazy)), "%")
print("Big forest     -> typical error:", round(percent_mistake(real, guesses)), "%",
      "| average mistake: Rs", round((real - guesses).abs().mean()),
      "| R2:", round(r2_score(yb_test, guesses_log), 2))

Lazy baseline  -> typical error: 62 %
Big forest     -> typical error: 36 % | average mistake: Rs 1282 | R2: 0.62
